In [2]:
import jax
import jax.numpy as jnp
import optax

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# prep
iris = load_iris()
X = iris.data
y = iris.target
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test = jnp.array(X_train), jnp.array(X_test)
y_train, y_test = jnp.array(y_train), jnp.array(y_test)

In [4]:
# initialize params
def init_params(input_dim, hidden_dim, output_dim):
	key1, key2 = jax.random.split(jax.random.PRNGKey(42))
	W1 = jax.random.normal(key1, (input_dim, hidden_dim)) * jnp.sqrt(2.0 / input_dim)  # Xavier initialization
	b1 = jnp.zeros(hidden_dim)
	W2 = jax.random.normal(key2, (hidden_dim, output_dim)) * jnp.sqrt(2.0 / hidden_dim)
	b2 = jnp.zeros(output_dim)
	return W1, b1, W2, b2

# model (nn)
def model(params, X):
	W1, b1, W2, b2 = params
	X = jnp.dot(X, W1) + b1
	X = jax.nn.relu(X)
	X = jnp.dot(X, W2) + b2
	return X

# loss function (cross entropy)
def compute_loss(params, X, y):
	logits = model(params, X)
	loss = optax.softmax_cross_entropy(logits, jax.nn.one_hot(y, num_classes=3)).mean()
	return loss

# update params
@jax.jit
def update(params, opt_state, X, y):
	grads = jax.grad(compute_loss)(params, X, y)
	updates, new_opt_state = optimizer.update(grads, opt_state)
	new_params = optax.apply_updates(params, updates)
	return new_params, new_opt_state

In [5]:
# define
params = init_params(input_dim=4, hidden_dim=16, output_dim=3)
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(params)

# train
epochs = 100
for epoch in range(epochs):
	params, opt_state = update(params, opt_state, X_train, y_train)

	if (epoch + 1) % 10 == 0 or epoch == 0:
		loss = compute_loss(params, X_train, y_train)
		print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss:.4f}")

Epoch [1/100], Loss: 1.6113
Epoch [10/100], Loss: 0.7773
Epoch [20/100], Loss: 0.4677
Epoch [30/100], Loss: 0.3706
Epoch [40/100], Loss: 0.3154
Epoch [50/100], Loss: 0.2728
Epoch [60/100], Loss: 0.2353
Epoch [70/100], Loss: 0.2002
Epoch [80/100], Loss: 0.1682
Epoch [90/100], Loss: 0.1381
Epoch [100/100], Loss: 0.1153


In [6]:
# evaluate
logits = model(params, X_test)
y_pred = jax.nn.softmax(logits, axis=1)
accuracy = jnp.mean(jnp.argmax(y_pred, axis=1) == y_test)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 100.00%
